In [5]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# SET PATH - GOOGLE COLAB
# ============================================================

DATA = Path("/content")
OUTPUT = Path("/content/output")
OUTPUT.mkdir(exist_ok=True)

print("=" * 70)
print("TECHTROVE DATA INTEGRATION PIPELINE")
print("=" * 70)

# ============================================================
# TODO 1: Extract CSV, Excel, JSON
# ============================================================

print("\n========== TODO 1: EXTRACT ==========")

# Orders
orders_jan = pd.read_csv(
    DATA / "orders_2026_01.csv"
)

orders_feb = pd.read_csv(
    DATA / "orders_2026_02.csv"
)

# Customers
customers = pd.read_csv(
    DATA / "customers_crm.csv"
)

# Products
products = pd.read_excel(
    DATA / "product_master.xlsx"
)

# Payments
with open(
    DATA / "payments.json",
    "r",
    encoding="utf-8"
) as f:
    payments_raw = json.load(f)

payments = pd.json_normalize(payments_raw)

payments = payments.rename(
    columns={
        "payment.method": "payment_method",
        "payment.status": "payment_status"
    }
)

print("orders_2026_01 :", orders_jan.shape)
print("orders_2026_02 :", orders_feb.shape)
print("customers      :", customers.shape)
print("products       :", products.shape)
print("payments       :", payments.shape)


# ============================================================
# PROFILE FUNCTION
# ============================================================

def profile_data(df, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Shape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nDtype:")
    print(df.dtypes)

    print("\nMissing:")
    display(
        df.isna().sum().to_frame("missing")
    )

    print("\nDuplicate rows:")
    print(
        df.duplicated().sum()
    )

    print("\nSample:")
    display(
        df.head()
    )


# ============================================================
# PROFILE BEFORE CLEANING
# ============================================================

profile_data(
    orders_jan,
    "orders_2026_01.csv"
)

profile_data(
    orders_feb,
    "orders_2026_02.csv"
)

profile_data(
    customers,
    "customers_crm.csv"
)

profile_data(
    products,
    "product_master.xlsx"
)

profile_data(
    payments,
    "payments.json"
)


# ============================================================
# DATA QUALITY REPORT
# ============================================================

quality_report = []


def add_quality(
    stage,
    metric,
    value,
    reason
):
    quality_report.append({
        "stage": stage,
        "metric": metric,
        "value": value,
        "reason": reason
    })


# Raw counts
raw_orders = (
    len(orders_jan)
    +
    len(orders_feb)
)

raw_order_ids = pd.concat(
    [
        orders_jan["order_id"],
        orders_feb["order_id"]
    ],
    ignore_index=True
)

raw_duplicate_order_ids = int(
    raw_order_ids.duplicated().sum()
)

raw_duplicate_customer_ids = int(
    customers["customer_id"]
    .duplicated()
    .sum()
)

raw_duplicate_payment_ids = int(
    payments["payment_id"]
    .duplicated()
    .sum()
)

raw_missing_customer_email = int(
    customers["email"].isna().sum()
)

raw_missing_jan_price = int(
    orders_jan["unit_price"].isna().sum()
)

raw_missing_feb_price = int(
    orders_feb["unit_price"].isna().sum()
)

add_quality(
    "raw",
    "raw_orders",
    raw_orders,
    "January + February"
)

add_quality(
    "raw",
    "duplicate_order_ids",
    raw_duplicate_order_ids,
    "duplicate order_id"
)

add_quality(
    "raw",
    "duplicate_customer_ids",
    raw_duplicate_customer_ids,
    "duplicate customer_id"
)

add_quality(
    "raw",
    "duplicate_payment_ids",
    raw_duplicate_payment_ids,
    "duplicate payment_id"
)

add_quality(
    "raw",
    "missing_customer_emails",
    raw_missing_customer_email,
    "email missing"
)

add_quality(
    "raw",
    "missing_unit_price_january",
    raw_missing_jan_price,
    "January unit_price missing"
)

add_quality(
    "raw",
    "missing_unit_price_february",
    raw_missing_feb_price,
    "February unit_price missing"
)


# ============================================================
# TODO 2: SCHEMA ALIGNMENT + CONCAT
# ============================================================

print(
    "\n========== TODO 2: SCHEMA ALIGNMENT + CONCAT =========="
)


# --------------------------
# January
# --------------------------

orders_jan = orders_jan.rename(
    columns={
        "order_date": "order_datetime"
    }
)

orders_jan["order_datetime"] = pd.to_datetime(
    orders_jan["order_datetime"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

orders_jan["discount"] = pd.to_numeric(
    orders_jan["discount"],
    errors="coerce"
)


# --------------------------
# February
# --------------------------

orders_feb = orders_feb.rename(
    columns={
        "ordered_at": "order_datetime",
        "qty": "quantity",
        "discount_pct": "discount"
    }
)

orders_feb["order_datetime"] = pd.to_datetime(
    orders_feb["order_datetime"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

orders_feb["discount"] = (
    orders_feb["discount"]
    .astype(str)
    .str.strip()
    .str.replace("%", "", regex=False)
)

orders_feb["discount"] = (
    pd.to_numeric(
        orders_feb["discount"],
        errors="coerce"
    ) / 100
)


print("\nJanuary schema:")
print(
    orders_jan.columns.tolist()
)

print("\nFebruary schema:")
print(
    orders_feb.columns.tolist()
)


# --------------------------
# CONCAT
# --------------------------

orders = pd.concat(
    [
        orders_jan,
        orders_feb
    ],
    ignore_index=True
)

print(
    "\nOrders after concat:",
    len(orders)
)

add_quality(
    "concat",
    "orders_after_concat",
    len(orders),
    "pd.concat(ignore_index=True)"
)


# ============================================================
# TODO 3: CLEAN / STANDARDIZE / DEDUPLICATE
# ============================================================

print(
    "\n========== TODO 3: CLEANING =========="
)


# --------------------------
# Convert data types
# --------------------------

orders["quantity"] = pd.to_numeric(
    orders["quantity"],
    errors="coerce"
)

orders["unit_price"] = pd.to_numeric(
    orders["unit_price"],
    errors="coerce"
)

orders["discount"] = pd.to_numeric(
    orders["discount"],
    errors="coerce"
)


# --------------------------
# Trim text
# --------------------------

for col in [
    "order_id",
    "customer_id",
    "product_id",
    "channel"
]:

    orders[col] = (
        orders[col]
        .astype("string")
        .str.strip()
    )


# --------------------------
# Deduplicate Orders
# --------------------------

before_dedup = len(orders)

duplicate_orders_removed = int(
    orders["order_id"]
    .duplicated()
    .sum()
)

orders = (
    orders
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .reset_index(drop=True)
)

print(
    "Before dedup:",
    before_dedup
)

print(
    "Removed:",
    duplicate_orders_removed
)

print(
    "After dedup:",
    len(orders)
)

add_quality(
    "cleaned",
    "duplicate_orders_removed",
    duplicate_orders_removed,
    "เก็บ order_id ล่าสุด"
)


# ============================================================
# CLEAN CUSTOMER
# ============================================================

customers["customer_id"] = (
    customers["customer_id"]
    .astype("string")
    .str.strip()
)

customers["full_name"] = (
    customers["full_name"]
    .astype("string")
    .str.strip()
)

customers["email"] = (
    customers["email"]
    .astype("string")
    .str.strip()
    .str.lower()
)

customers["province"] = (
    customers["province"]
    .astype("string")
    .str.strip()
)

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"],
    errors="coerce"
)


# ============================================================
# STANDARDIZE PROVINCE
# ============================================================

province_map = {
    "Bangkok": "กรุงเทพมหานคร",
    "กรุงเทพ": "กรุงเทพมหานคร",
    "กรุงเทพฯ": "กรุงเทพมหานคร",
    "กทม.": "กรุงเทพมหานคร",

    "Chonburi": "ชลบุรี",
    "ชลบุรี ": "ชลบุรี",

    "Chiang Mai": "เชียงใหม่",
    "เชียงใหม่ ": "เชียงใหม่",

    "Khon Kaen": "ขอนแก่น",
    "ขอนเเก่น": "ขอนแก่น",
    "ขอนแก่น ": "ขอนแก่น",

    "Rayong": "ระยอง",
    "ระยอง ": "ระยอง",

    "Phuket": "ภูเก็ต",
    "ภูเก็ต ": "ภูเก็ต"
}

customers["province"] = (
    customers["province"]
    .replace(province_map)
)


# ============================================================
# DEDUP CUSTOMER
# ============================================================

duplicate_customer_removed = int(
    customers["customer_id"]
    .duplicated()
    .sum()
)

customers = (
    customers
    .drop_duplicates(
        subset="customer_id",
        keep="last"
    )
    .reset_index(drop=True)
)

print(
    "Customer duplicate removed:",
    duplicate_customer_removed
)

add_quality(
    "cleaned",
    "duplicate_customer_rows_removed",
    duplicate_customer_removed,
    "เก็บ customer_id ล่าสุด"
)


# ============================================================
# CLEAN PRODUCT
# ============================================================

products["product_id"] = (
    products["product_id"]
    .astype("string")
    .str.strip()
)

products["product_name"] = (
    products["product_name"]
    .astype("string")
    .str.strip()
)

products["category"] = (
    products["category"]
    .astype("string")
    .str.strip()
)

products["standard_price"] = pd.to_numeric(
    products["standard_price"],
    errors="coerce"
)

products["active_flag"] = (
    products["active_flag"]
    .astype("string")
    .str.strip()
    .str.upper()
)


# ============================================================
# CLEAN PAYMENT
# ============================================================

payments["payment_id"] = (
    payments["payment_id"]
    .astype("string")
    .str.strip()
)

payments["order_id"] = (
    payments["order_id"]
    .astype("string")
    .str.strip()
)

payments["payment_method"] = (
    payments["payment_method"]
    .astype("string")
    .str.strip()
)

payments["payment_status"] = (
    payments["payment_status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

payments["paid_at"] = pd.to_datetime(
    payments["paid_at"],
    errors="coerce"
)


# --------------------------
# Deduplicate Payment ID
# --------------------------

duplicate_payment_removed = int(
    payments["payment_id"]
    .duplicated()
    .sum()
)

payments = (
    payments
    .drop_duplicates(
        subset="payment_id",
        keep="last"
    )
    .reset_index(drop=True)
)

add_quality(
    "cleaned",
    "duplicate_payment_ids_removed",
    duplicate_payment_removed,
    "เก็บ payment_id ล่าสุด"
)


# ============================================================
# TODO 4: MERGE CUSTOMER / PRODUCT / PAYMENT
# ============================================================

print(
    "\n========== TODO 4: INTEGRATION =========="
)


# --------------------------
# Customer Merge
# --------------------------

fact = orders.merge(
    customers[
        [
            "customer_id",
            "full_name",
            "email",
            "province",
            "signup_date"
        ]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="_customer_match"
)

print("\nCustomer Match:")
print(
    fact["_customer_match"]
    .value_counts()
)

customer_unmatched_rows = int(
    (
        fact["_customer_match"]
        == "left_only"
    ).sum()
)

add_quality(
    "integration",
    "unmatched_customer_rows",
    customer_unmatched_rows,
    "customer_id ไม่พบใน Master"
)


# --------------------------
# Product Merge
# --------------------------

fact = fact.merge(
    products[
        [
            "product_id",
            "product_name",
            "category",
            "standard_price",
            "active_flag"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="_product_match"
)

print("\nProduct Match:")
print(
    fact["_product_match"]
    .value_counts()
)

product_unmatched_rows = int(
    (
        fact["_product_match"]
        == "left_only"
    ).sum()
)

add_quality(
    "integration",
    "unmatched_product_rows",
    product_unmatched_rows,
    "product_id ไม่พบใน Master"
)


# --------------------------
# Payment
# --------------------------

payments = (
    payments
    .sort_values(
        ["order_id", "paid_at"]
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    .reset_index(drop=True)
)

fact = fact.merge(
    payments[
        [
            "order_id",
            "payment_id",
            "payment_method",
            "payment_status",
            "paid_at"
        ]
    ],
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator="_payment_match"
)

print("\nPayment Match:")
print(
    fact["_payment_match"]
    .value_counts()
)

payment_unmatched_rows = int(
    (
        fact["_payment_match"]
        == "left_only"
    ).sum()
)

add_quality(
    "integration",
    "unmatched_payment_rows",
    payment_unmatched_rows,
    "order_id ไม่มี payment"
)


# ============================================================
# TODO 5: VALIDATE BUSINESS RULES
# ============================================================

print(
    "\n========== TODO 5: VALIDATION =========="
)


valid_quantity = (
    fact["quantity"].notna()
    &
    (fact["quantity"] > 0)
)

valid_unit_price = (
    fact["unit_price"].notna()
    &
    (fact["unit_price"] > 0)
)

valid_discount = (
    fact["discount"].notna()
    &
    fact["discount"].between(
        0,
        1,
        inclusive="both"
    )
)

valid_customer = (
    fact["_customer_match"]
    == "both"
)

valid_product = (
    fact["_product_match"]
    == "both"
)

valid_payment = (
    fact["payment_status"]
    == "PAID"
)


invalid_quantity_rows = int(
    (~valid_quantity).sum()
)

invalid_unit_price_rows = int(
    (~valid_unit_price).sum()
)

invalid_discount_rows = int(
    (~valid_discount).sum()
)


print(
    "Quantity > 0:",
    int(valid_quantity.sum())
)

print(
    "Unit price > 0:",
    int(valid_unit_price.sum())
)

print(
    "Discount 0-1:",
    int(valid_discount.sum())
)

print(
    "Customer matched:",
    int(valid_customer.sum())
)

print(
    "Product matched:",
    int(valid_product.sum())
)

print(
    "Payment PAID:",
    int(valid_payment.sum())
)


add_quality(
    "validation",
    "invalid_quantity_rows",
    invalid_quantity_rows,
    "quantity ต้อง > 0"
)

add_quality(
    "validation",
    "invalid_unit_price_rows",
    invalid_unit_price_rows,
    "unit_price ต้อง > 0"
)

add_quality(
    "validation",
    "invalid_discount_rows",
    invalid_discount_rows,
    "discount ต้องอยู่ 0 ถึง 1"
)


# ============================================================
# NET SALES
# ============================================================

valid_sales = (
    valid_quantity
    &
    valid_unit_price
    &
    valid_discount
    &
    valid_customer
    &
    valid_product
    &
    valid_payment
)

fact["net_sales"] = 0.0

fact.loc[
    valid_sales,
    "net_sales"
] = (
    fact.loc[
        valid_sales,
        "quantity"
    ]
    *
    fact.loc[
        valid_sales,
        "unit_price"
    ]
    *
    (
        1
        -
        fact.loc[
            valid_sales,
            "discount"
        ]
    )
)


# ============================================================
# TODO 6: DIMENSION + FACT
# ============================================================

print(
    "\n========== TODO 6: DIMENSION / FACT =========="
)


# --------------------------
# Fact Sales
# --------------------------

fact_sales = (
    fact.loc[
        valid_sales
    ]
    .copy()
)

fact_sales["net_sales"] = (
    fact_sales["net_sales"]
    .round(2)
)

fact_sales = fact_sales[
    [
        "order_id",
        "order_datetime",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount",
        "channel",
        "payment_id",
        "payment_method",
        "payment_status",
        "paid_at",
        "province",
        "category",
        "standard_price",
        "active_flag",
        "net_sales"
    ]
].reset_index(drop=True)


# --------------------------
# Dimension Customer
# --------------------------

dim_customer = customers[
    [
        "customer_id",
        "full_name",
        "email",
        "province",
        "signup_date"
    ]
].copy()


# --------------------------
# Dimension Product
# --------------------------

dim_product = products[
    [
        "product_id",
        "product_name",
        "category",
        "standard_price",
        "active_flag"
    ]
].copy()


# ============================================================
# SAVE FACT / DIM
# ============================================================

dim_customer.to_csv(
    OUTPUT / "dim_customer.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_product.to_csv(
    OUTPUT / "dim_product.csv",
    index=False,
    encoding="utf-8-sig"
)

fact_sales.to_csv(
    OUTPUT / "fact_sales.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# DATA QUALITY FINAL
# ============================================================

failed_payment_rows = int(
    (
        fact["payment_status"]
        == "FAILED"
    ).sum()
)

refunded_payment_rows = int(
    (
        fact["payment_status"]
        == "REFUNDED"
    ).sum()
)

total_net_sales = round(
    fact_sales["net_sales"].sum(),
    2
)

add_quality(
    "final",
    "valid_paid_sales",
    len(fact_sales),
    "ผ่านทุก Business Rules และ PAID"
)

add_quality(
    "final",
    "total_net_sales",
    total_net_sales,
    "sum(net_sales)"
)

add_quality(
    "final",
    "failed_payment_rows",
    failed_payment_rows,
    "ไม่ถือเป็นยอดขาย"
)

add_quality(
    "final",
    "refunded_payment_rows",
    refunded_payment_rows,
    "ไม่ถือเป็นยอดขาย"
)

add_quality(
    "final",
    "orders_after_dedup",
    len(orders),
    "หลัง deduplicate order_id"
)


data_quality_report = pd.DataFrame(
    quality_report
)

data_quality_report.to_csv(
    OUTPUT / "data_quality_report.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# TODO 7: SUMMARY
# ============================================================

print(
    "\n========== TODO 7: ANALYSIS =========="
)


# --------------------------
# Province
# --------------------------

summary_by_province = (
    fact_sales
    .groupby(
        "province",
        as_index=False
    )
    .agg(
        transactions=(
            "order_id",
            "nunique"
        ),
        total_quantity=(
            "quantity",
            "sum"
        ),
        net_sales=(
            "net_sales",
            "sum"
        )
    )
    .sort_values(
        "net_sales",
        ascending=False
    )
    .reset_index(drop=True)
)

summary_by_province["net_sales"] = (
    summary_by_province["net_sales"]
    .round(2)
)


# --------------------------
# Category
# --------------------------

summary_by_category = (
    fact_sales
    .groupby(
        "category",
        as_index=False
    )
    .agg(
        transactions=(
            "order_id",
            "nunique"
        ),
        total_quantity=(
            "quantity",
            "sum"
        ),
        net_sales=(
            "net_sales",
            "sum"
        )
    )
    .sort_values(
        "net_sales",
        ascending=False
    )
    .reset_index(drop=True)
)

summary_by_category["net_sales"] = (
    summary_by_category["net_sales"]
    .round(2)
)


# ============================================================
# SAVE SUMMARY
# ============================================================

summary_by_province.to_csv(
    OUTPUT / "summary_by_province.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_by_category.to_csv(
    OUTPUT / "summary_by_category.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# ANSWERS 6 QUESTIONS
# ============================================================

top_province = (
    summary_by_province.iloc[0]
)

top_category = (
    summary_by_category.iloc[0]
)


answers = f"""
TechTrove E-Commerce
Data Integration Analysis

1. หลังรวมไฟล์ orders มีจำนวน {raw_orders:,} แถว
   และเหลือ {len(orders):,} แถวหลังลบ duplicate
   โดยลบ duplicate ออก {duplicate_orders_removed:,} แถว

2. มี customer_id ที่ไม่พบใน Master Data
   จำนวน {customer_unmatched_rows:,} แถว

   มี product_id ที่ไม่พบใน Master Data
   จำนวน {product_unmatched_rows:,} แถว

3. มียอดขายที่ใช้ได้จริง
   จำนวน {len(fact_sales):,} ธุรกรรม

   ยอดขายสุทธิรวม
   {total_net_sales:,.2f} บาท

4. จังหวัดที่มียอดขายสุทธิสูงสุดคือ
   {top_province["province"]}

   ยอดขายสุทธิ
   {top_province["net_sales"]:,.2f} บาท

5. หมวดสินค้าที่มียอดขายสุทธิสูงสุดคือ
   {top_category["category"]}

   ยอดขายสุทธิ
   {top_category["net_sales"]:,.2f} บาท

6. หากสลับลำดับ merge ก่อน cleaning
   ความน่าเชื่อถือของข้อมูลอาจลดลง เนื่องจากข้อมูลดิบยังมี
   duplicate key, schema ที่แตกต่างกันระหว่างเดือน,
   รูปแบบข้อความที่ไม่เป็นมาตรฐาน และข้อมูลผิดกฎธุรกิจ

   การ merge ก่อน cleaning อาจทำให้เกิดการจับคู่ผิด
   จำนวนแถวเพิ่มจาก duplicate key หรือข้อมูล invalid
   ถูกนำไปใช้ในการวิเคราะห์

   ดังนั้นควรทำ Schema Alignment,
   Cleaning, Standardization, Deduplication
   และ Validation ก่อน Merge
   เพื่อควบคุม cardinality และตรวจสอบ unmatched keys
"""


with open(
    OUTPUT / "analysis_answers.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(answers)


# ============================================================
# FINAL DISPLAY
# ============================================================

print("\n")
print("=" * 70)
print("                    FINAL OUTPUT")
print("=" * 70)

print(
    f"Raw Orders              : {raw_orders:,}"
)

print(
    f"Orders After Dedup      : {len(orders):,}"
)

print(
    f"Duplicate Removed       : {duplicate_orders_removed:,}"
)

print(
    f"Customer Unmatched Rows : {customer_unmatched_rows:,}"
)

print(
    f"Product Unmatched Rows  : {product_unmatched_rows:,}"
)

print(
    f"Valid Paid Sales        : {len(fact_sales):,}"
)

print(
    f"Total Net Sales         : {total_net_sales:,.2f} บาท"
)

print(
    f"Top Province            : {top_province['province']}"
)

print(
    f"Top Province Sales      : {top_province['net_sales']:,.2f} บาท"
)

print(
    f"Top Category            : {top_category['category']}"
)

print(
    f"Top Category Sales      : {top_category['net_sales']:,.2f} บาท"
)


# ============================================================
# DISPLAY TABLES
# ============================================================

print("\n========== SUMMARY BY PROVINCE ==========")
display(
    summary_by_province
)

print("\n========== SUMMARY BY CATEGORY ==========")
display(
    summary_by_category
)

print("\n========== DATA QUALITY REPORT ==========")
display(
    data_quality_report
)

print("\n========== FACT SALES ==========")
display(
    fact_sales.head(10)
)

print("\n========== DIM CUSTOMER ==========")
display(
    dim_customer.head(10)
)

print("\n========== DIM PRODUCT ==========")
display(
    dim_product.head(10)
)


# ============================================================
# OUTPUT FILE CHECK
# ============================================================

print("\n========== OUTPUT FILES ==========")

output_files = [
    "dim_customer.csv",
    "dim_product.csv",
    "fact_sales.csv",
    "data_quality_report.csv",
    "summary_by_province.csv",
    "summary_by_category.csv",
    "analysis_answers.txt"
]

for filename in output_files:
    path = OUTPUT / filename

    print(
        filename,
        "->",
        "OK" if path.exists() else "NOT FOUND"
    )

print("\nOutput folder:", OUTPUT)
print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

TECHTROVE DATA INTEGRATION PIPELINE

========== TODO 1: EXTRACT ==========
orders_2026_01 : (361, 8)
orders_2026_02 : (391, 8)
customers      : (163, 5)
products       : (40, 5)
payments       : (752, 5)

orders_2026_01.csv
Shape:
(361, 8)

Columns:
['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'discount', 'channel']

Dtype:
order_id        object
order_date      object
customer_id     object
product_id      object
quantity         int64
unit_price     float64
discount       float64
channel         object
dtype: object

Missing:


,missing
order_id,0
order_date,0
customer_id,0
product_id,0
quantity,0
unit_price,1
discount,0
channel,0



Duplicate rows:
1

Sample:


,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,channel
0,ORD000001,2026-01-08 17:11:00,C0158,P039,2,940.5,0.00,Marketplace
1,ORD000002,2026-01-12 20:13:00,C0123,P038,1,12900.0,0.10,Mobile App
2,ORD000003,2026-01-26 14:00:00,C0119,P031,3,1415.5,0.05,Web
3,ORD000004,2026-01-18 16:18:00,C0065,P003,2,25900.0,0.00,Mobile App
4,ORD000005,2026-01-08 17:43:00,C0129,P007,2,23310.0,0.05,Mobile App



orders_2026_02.csv
Shape:
(391, 8)

Columns:
['order_id', 'ordered_at', 'customer_id', 'product_id', 'qty', 'unit_price', 'discount_pct', 'channel']

Dtype:
order_id         object
ordered_at       object
customer_id      object
product_id       object
qty               int64
unit_price      float64
discount_pct     object
channel          object
dtype: object

Missing:


...


========== DIM CUSTOMER ==========


,customer_id,full_name,email,province,signup_date
0,C0001,ลูกค้า 001,customer001@example.com,ชลบุรี,2024-07-29
1,C0002,ลูกค้า 002,customer002@example.com,ชลบุรี,2025-01-04
2,C0003,ลูกค้า 003,customer003@example.com,ขอนแก่น,2025-10-10
3,C0004,ลูกค้า 004,customer004@example.com,กรุงเทพมหานคร,2024-07-11
4,C0005,ลูกค้า 005,customer005@example.com,ระยอง,2025-06-27
5,C0006,ลูกค้า 006,customer006@example.com,ระยอง,2024-07-13
6,C0007,ลูกค้า 007,customer007@example.com,กรุงเทพมหานคร,2025-09-27
7,C0008,ลูกค้า 008,customer008@example.com,ขอนแก่น,2025-05-27
8,C0009,ลูกค้า 009,customer009@example.com,ภูเก็ต,2024-09-06
9,C0010,ลูกค้า 010,customer010@example.com,ชลบุรี,2024-02-04



========== DIM PRODUCT ==========


,product_id,product_name,category,standard_price,active_flag
0,P001,Notebook Model 01,Notebook,299,Y
1,P002,Smartphone Model 02,Smartphone,1490,Y
2,P003,Smartphone Model 03,Smartphone,25900,Y
3,P004,Smart Home Model 04,Smart Home,1490,Y
4,P005,Smartphone Model 05,Smartphone,299,Y
5,P006,Accessory Model 06,Accessory,299,Y
6,P007,Notebook Model 07,Notebook,25900,Y
7,P008,Accessory Model 08,Accessory,499,Y
8,P009,Notebook Model 09,Notebook,299,Y
9,P010,Smart Home Model 10,Smart Home,990,Y



========== OUTPUT FILES ==========
dim_customer.csv -> OK
dim_product.csv -> OK
fact_sales.csv -> OK
data_quality_report.csv -> OK
summary_by_province.csv -> OK
summary_by_category.csv -> OK
analysis_answers.txt -> OK

Output folder: /content/output
PIPELINE COMPLETE
